# RetailPulse Phase 1 - Exploratory Data Analysis

This notebook explores the cleaned retail transaction data through descriptive analytics, trend analysis, and professional visualizations.

In [ ]:
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.append(str(project_root / 'src'))
import pandas as pd
from config import PROCESSED_DATA_PATH, FIGURES_DIR
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv(PROCESSED_DATA_PATH, parse_dates=['InvoiceDate'])
df = df.sort_values('InvoiceDate').reset_index(drop=True)
df.head()


In [ ]:
# Helper to save plots
def save_plot(name, fig):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / name, dpi=300, bbox_inches='tight')
    plt.close(fig)

# 1. Sales Trend
sales_daily = df.set_index('InvoiceDate').resample('D')['Revenue'].sum()
fig, ax = plt.subplots(figsize=(12, 4))
sns.lineplot(x=sales_daily.index, y=sales_daily.values, ax=ax, color='#2c7fb8')
ax.set_title('Sales Trend')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue')
save_plot('01_sales_trend.png', fig)

# 2. Monthly Trend
monthly = df.groupby(df['InvoiceDate'].dt.to_period('M')).agg(TotalRevenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'))
monthly.index = monthly.index.to_timestamp()
fig, ax = plt.subplots(figsize=(12, 4))
sns.lineplot(x=monthly.index, y=monthly['TotalRevenue'], ax=ax, color='#f03b20')
ax.set_title('Monthly Revenue Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
save_plot('02_monthly_trend.png', fig)

# 3. Weekly Trend
weekly = df.groupby('Week')['Revenue'].sum()
fig, ax = plt.subplots(figsize=(12, 4))
sns.lineplot(x=weekly.index, y=weekly.values, ax=ax, color='#31a354')
ax.set_title('Weekly Revenue Trend')
ax.set_xlabel('Week')
ax.set_ylabel('Revenue')
save_plot('03_weekly_trend.png', fig)

# 4. Hourly Trend
hourly = df.groupby('Hour')['Revenue'].sum()
fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(x=hourly.index, y=hourly.values, ax=ax, palette='viridis')
ax.set_title('Hourly Revenue Trend')
ax.set_xlabel('Hour')
ax.set_ylabel('Revenue')
save_plot('04_hourly_trend.png', fig)

# 5. Country Analysis
country_sales = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(x=country_sales.index, y=country_sales.values, ax=ax, palette='mako')
ax.set_title('Top Countries by Revenue')
ax.set_xlabel('Country')
ax.set_ylabel('Revenue')
ax.tick_params(axis='x', rotation=45)
save_plot('05_country_analysis.png', fig)

# 6. Top Products
products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(x=products.index, y=products.values, ax=ax, palette='viridis')
ax.set_title('Top Products')
ax.set_xlabel('Product')
ax.set_ylabel('Revenue')
ax.tick_params(axis='x', rotation=45)
save_plot('06_top_products.png', fig)

# 7. Bottom Products
bottom_products = df.groupby('Description')['Revenue'].sum().sort_values(ascending=True).head(10)
fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(x=bottom_products.index, y=bottom_products.values, ax=ax, palette='rocket')
ax.set_title('Bottom Products')
ax.set_xlabel('Product')
ax.set_ylabel('Revenue')
ax.tick_params(axis='x', rotation=45)
save_plot('07_bottom_products.png', fig)

# 8. Top Customers
customers = df.groupby('CustomerID')['Revenue'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(x=customers.index.astype(int).astype(str), y=customers.values, ax=ax, palette='deep')
ax.set_title('Top Customers by Revenue')
ax.set_xlabel('CustomerID')
ax.set_ylabel('Revenue')
save_plot('08_top_customers.png', fig)

# 9. Revenue Distribution
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df['Revenue'], bins=40, kde=True, ax=ax, color='#1f77b4')
ax.set_title('Revenue Distribution')
ax.set_xlabel('Revenue')
ax.set_ylabel('Frequency')
save_plot('09_revenue_distribution.png', fig)

# 10. Quantity Distribution
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df['Quantity'], bins=40, kde=True, ax=ax, color='#ff7f0e')
ax.set_title('Quantity Distribution')
ax.set_xlabel('Quantity')
ax.set_ylabel('Frequency')
save_plot('10_quantity_distribution.png', fig)

# 11. Basket Size
basket_size = df.groupby('InvoiceNo').size()
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(basket_size, bins=30, kde=True, ax=ax, color='#2ca02c')
ax.set_title('Basket Size Distribution')
ax.set_xlabel('Items per Basket')
ax.set_ylabel('Transactions')
save_plot('11_basket_size.png', fig)

# 12. Average Order Value
order_value = df.groupby('InvoiceNo')['Revenue'].sum()
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(order_value, bins=40, kde=True, ax=ax, color='#9467bd')
ax.set_title('Average Order Value Distribution')
ax.set_xlabel('Order Value')
ax.set_ylabel('Orders')
save_plot('12_average_order_value.png', fig)

# 13. Customer Frequency
customer_frequency = df.groupby('CustomerID').size()
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(customer_frequency, bins=30, kde=True, ax=ax, color='#8c564b')
ax.set_title('Customer Frequency Distribution')
ax.set_xlabel('Orders per Customer')
ax.set_ylabel('Customers')
save_plot('13_customer_frequency.png', fig)

# 14. Product Frequency
product_frequency = df.groupby('Description').size()
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(product_frequency, bins=30, kde=True, ax=ax, color='#e377c2')
ax.set_title('Product Frequency Distribution')
ax.set_xlabel('Orders per Product')
ax.set_ylabel('Products')
save_plot('14_product_frequency.png', fig)

# 15. Monthly Heatmap
a_month_country = df.pivot_table(index='Month', columns='Country', values='Revenue', aggfunc='sum').fillna(0)
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(a_month_country.head(12), cmap='Blues', ax=ax)
ax.set_title('Revenue Heatmap by Month and Country')
ax.set_xlabel('Country')
ax.set_ylabel('Month')
save_plot('15_month_country_heatmap.png', fig)

# 16. Correlation Matrix
num_cols = ['Quantity','UnitPrice','Revenue','Profit','AverageBasketValue','CLV','CustomerTenure','CustomerFrequency','MonthlyRevenue','MonthlyOrderCount','AverageOrderValue']
correlation = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Matrix')
save_plot('16_correlation_matrix.png', fig)

# 17. Revenue Boxplot by Country
fig, ax = plt.subplots(figsize=(12, 4))
sns.boxplot(data=df, x='Country', y='Revenue', ax=ax)
ax.set_title('Revenue Boxplot by Country')
ax.tick_params(axis='x', rotation=45)
save_plot('17_country_boxplot.png', fig)

# 18. Revenue by Weekday
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=df, x='DayOfWeek', y='Revenue', ax=ax, order=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
ax.set_title('Revenue by Weekday')
save_plot('18_weekday_boxplot.png', fig)

# 19. Revenue Violin by Month
fig, ax = plt.subplots(figsize=(10, 4))
sns.violinplot(data=df, x='Month', y='Revenue', ax=ax)
ax.set_title('Revenue Violin by Month')
save_plot('19_month_violin.png', fig)

# 20. Revenue Violin by Weekend
fig, ax = plt.subplots(figsize=(10, 4))
sns.violinplot(data=df, x='IsWeekend', y='Revenue', ax=ax)
ax.set_title('Revenue Violin by Weekend')
save_plot('20_weekend_violin.png', fig)

# 21. Pareto Analysis
pareto = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False)
pareto_cum = pareto.cumsum() / pareto.sum() * 100
fig, ax = plt.subplots(figsize=(12, 4))
pareto_cum.head(20).plot(ax=ax, marker='o')
ax.set_title('Pareto Analysis')
ax.set_xlabel('Products')
ax.set_ylabel('Cumulative Revenue %')
save_plot('21_pareto_analysis.png', fig)

# 22. ABC Analysis
abc_labels = pd.Series(['A'] * len(pareto), index=pareto.index)
abc_cum = pareto.cumsum() / pareto.sum()
abc_labels.iloc[abc_cum[abc_cum > 0.8].index] = 'B'
abc_labels.iloc[abc_cum[abc_cum > 0.95].index] = 'C'
fig, ax = plt.subplots(figsize=(8, 4))
abc_labels.value_counts().reindex(['A','B','C']).plot(kind='bar', ax=ax, color=['#1f77b4','#ff7f0e','#2ca02c'])
ax.set_title('ABC Analysis')
ax.set_xlabel('Class')
ax.set_ylabel('Products')
save_plot('22_abc_analysis.png', fig)

# 23. Seasonal Analysis
seasonal = df.groupby(['Season','Year'])['Revenue'].sum().reset_index()
fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=seasonal, x='Season', y='Revenue', hue='Year', ax=ax)
ax.set_title('Seasonal Revenue Analysis')
save_plot('23_seasonal_analysis.png', fig)

# 24. RFM Preview
rfm = df.groupby('CustomerID').agg(Recency=('InvoiceDate','max'), Frequency=('InvoiceNo','nunique'), Monetary=('Revenue','sum'))
rfm['Recency'] = (df['InvoiceDate'].max() - rfm['Recency']).dt.days
fig, ax = plt.subplots(figsize=(10, 4))
sns.scatterplot(data=rfm.sample(1000, random_state=42), x='Frequency', y='Monetary', hue='Recency', ax=ax)
ax.set_title('RFM Preview')
save_plot('24_rfm_preview.png', fig)

# 25. CLV Distribution
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df['CLV'], bins=30, kde=True, ax=ax, color='#17becf')
ax.set_title('CLV Distribution')
ax.set_xlabel('CLV')
ax.set_ylabel('Customers')
save_plot('25_clv_distribution.png', fig)

# 26. Country Comparison
country_summary = df.groupby('Country').agg(Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'))
fig, ax = plt.subplots(figsize=(12, 4))
sns.barplot(data=country_summary.reset_index().head(10), x='Country', y='Revenue', ax=ax)
ax.set_title('Country Revenue Comparison')
ax.tick_params(axis='x', rotation=45)
save_plot('26_country_comparison.png', fig)

# 27. Revenue by Month
month_summary = df.groupby('Month')['Revenue'].sum()
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=month_summary.index, y=month_summary.values, ax=ax, palette='Set2')
ax.set_title('Revenue by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
save_plot('27_revenue_by_month.png', fig)

# 28. Revenue by Weekday
weekday_summary = df.groupby('DayOfWeek')['Revenue'].sum().reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=weekday_summary.index, y=weekday_summary.values, ax=ax, palette='pastel')
ax.set_title('Revenue by Weekday')
ax.set_xlabel('Weekday')
ax.set_ylabel('Revenue')
save_plot('28_revenue_by_weekday.png', fig)

# 29. Product Performance
product_performance = df.groupby('Description').agg(Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'), Customers=('CustomerID','nunique'))
fig, ax = plt.subplots(figsize=(12, 4))
sns.scatterplot(data=product_performance.head(100).reset_index(), x='Orders', y='Revenue', size='Customers', ax=ax)
ax.set_title('Product Performance')
save_plot('29_product_performance.png', fig)

# 30. Customer Performance
customer_performance = df.groupby('CustomerID').agg(Revenue=('Revenue','sum'), Orders=('InvoiceNo','nunique'), Products=('Description','nunique'))
fig, ax = plt.subplots(figsize=(12, 4))
sns.scatterplot(data=customer_performance.head(100).reset_index(), x='Orders', y='Revenue', size='Products', ax=ax)
ax.set_title('Customer Performance')
save_plot('30_customer_performance.png', fig)

# 31. Business Insights
summary_text = f"Total revenue: {df['Revenue'].sum():,.2f}; Customers: {df['CustomerID'].nunique()}; Products: {df['Description'].nunique()}; Countries: {df['Country'].nunique()}"
with open(FIGURES_DIR.parent / 'eda_summary.md', 'w', encoding='utf-8') as handle:
    handle.write('# EDA Summary\n\n')
    handle.write(summary_text + '\n')

# 32. Executive Summary
fig, ax = plt.subplots(figsize=(10, 4))
ax.text(0.5, 0.5, summary_text, ha='center', va='center', fontsize=12)
ax.axis('off')
ax.set_title('Executive Summary')
save_plot('32_executive_summary.png', fig)

print('Generated 32 visualizations and saved them in', FIGURES_DIR)
